# Fenix Music — QLoRA Training Notebook

تجربة سريعة لتدريب عقل **Fenix Music** على Google Colab بدون Modal.

- يفترض تشغيل الـNotebook على **GPU** من Runtime → Change runtime type.
- T4 16GB: استخدم `max_length=2048` وepochs=2.
- Colab A100/L4: يمكنك رفع `max_length` إلى 4096 وepochs=3.
- لا تضع HF_TOKEN في كود عام. استخدم Colab Secrets باسم `HF_TOKEN` أو أدخله عند الطلب.

> هذه التجربة تنشئ LoRA Adapter، ولا تحتاج إلى Modal.

In [ ]:
import importlib.util

if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Open this notebook in Google Colab, not ordinary Python.")

!pip -q install -U "transformers>=4.51" "peft>=0.11" "trl>=0.9" datasets accelerate bitsandbytes huggingface_hub safetensors

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. In Colab choose Runtime → Change runtime type → GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 1) تحميل بيانات Fenix Music

يجرب الـNotebook تحميل `data.jsonl` من فرع `main` في مستودع Fenix-ai. إذا لم يكن الملف متاحاً، سيطلب رفعه يدوياً.

In [ ]:
import json
import urllib.request
from pathlib import Path
from google.colab import files

DATA_PATH = Path("/content/fenix-music-data.jsonl")
DATA_URL = "https://raw.githubusercontent.com/hakarikenji/Fenix-ai/main/fenix-music/training/data.jsonl"
try:
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Downloaded dataset:", DATA_PATH)
except Exception as exc:
    print("Automatic download failed:", exc)
    print("Upload fenix-music/training/data.jsonl here:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No dataset uploaded.")
    DATA_PATH = Path(next(iter(uploaded)))

rows = [json.loads(line) for line in DATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assert rows, "Dataset is empty"
assert all([m.get("role") for m in row.get("messages", [])] == ["system", "user", "assistant"] for row in rows)
print("MUSIC_DATA_OK rows=", len(rows))

## 2) تسجيل Hugging Face — اختياري للنشر

إذا أردت رفع الـAdapter إلى `Hakari66684/fenix-music-lora`، أضف Secret في Colab باسم `HF_TOKEN`.
إذا لم ترغب في الرفع الآن، اترك هذه الخلية فارغة.

In [ ]:
import getpass
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    answer = input("Paste HF_TOKEN to enable upload (press Enter to skip): ").strip()
    hf_token = answer or None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face login OK")
else:
    print("Upload disabled; local adapter will still be saved.")

## 3) تحميل Qwen3-4B وإعداد QLoRA

سنستخدم NF4 + double quantization وLoRA rank 32 مع projections واسعة في attention وMLP.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("Loading", BASE_MODEL)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
print("Base model + QLoRA preparation complete")

## 4) بدء التدريب

اضبط `MAX_LENGTH=4096` فقط إذا كانت الذاكرة كافية. ابدأ بـ 2048 على T4.

In [ ]:
from trl import SFTConfig, SFTTrainer

MAX_LENGTH = 2048
EPOCHS = 2
lora = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    use_rslora=True,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
common = dict(
    output_dir="/content/fenix-music-checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=EPOCHS,
    learning_rate=1e-4,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    logging_steps=5,
    save_strategy="epoch",
    report_to=[],
    seed=42,
)
try:
    training_args = SFTConfig(max_length=MAX_LENGTH, packing=False, **common)
except TypeError:
    training_args = SFTConfig(max_seq_length=MAX_LENGTH, packing=False, **common)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=Dataset.from_list([{"messages": row["messages"]} for row in rows]),
    peft_config=lora,
    processing_class=tokenizer,
)
trainer.train()
adapter_dir = "/content/fenix-music-adapter"
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("LOCAL_ADAPTER_SAVED", adapter_dir)

## 5) حفظ Adapter كـZIP

هذا الملف هو نفس الاسم الذي تتوقعه نسخ HF Space الحالية.

In [ ]:
import zipfile
from pathlib import Path

adapter_dir = Path("/content/fenix-music-adapter")
zip_path = Path("/content/fenix-music-adapter.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(adapter_dir.rglob("*")):
        if file.is_file():
            archive.write(file, file.relative_to(adapter_dir.parent))
print("ZIP_SAVED", zip_path, zip_path.stat().st_size, "bytes")

## 6) رفع Adapter إلى Hugging Face — اختياري

يتطلب تشغيل الخلية أن تكون قد سجّلت الدخول في خلية HF_TOKEN أعلاه.

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "Hakari66684/fenix-music-lora"
if hf_token:
    HfApi(token=hf_token).upload_file(
        path_or_fileobj="/content/fenix-music-adapter.zip",
        path_in_repo="fenix-music-adapter.zip",
        repo_id=REPO_ID,
        repo_type="model",
    )
    print("PUBLISHED", REPO_ID + "/fenix-music-adapter.zip")
else:
    print("No token: download /content/fenix-music-adapter.zip manually.")

## 7) اختبار سريع للـAdapter

هذا الاختبار يولّد lyrics من نفس الـbase + adapter، لكنه ليس نشراً دائماً.

In [ ]:
prompt = "Write a dark Moroccan Darija phonk chorus about a midnight escape. Output only structured lyrics."
text = tokenizer.apply_chat_template(
    [{"role": "system", "content": "You are Fenix Music."}, {"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=350,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))